# E-Commerce Order Analytics System
### Intern Mini Project

This notebook contains the complete solution for processing, cleaning, analyzing, and reporting e-commerce order data using Python and MySQL 8.0+.

**Project Pipeline:**
1. Data Generation (creating sample datasets with intentional data quality issues)
2. Data Cleaning & Validation (handling dates, missing values, email validation, referential integrity)
3. SQL Analysis (executing 16 analytical queries using CTEs, window functions, and aggregations)
4. CLI Reporting Tool (dynamic summary report with period-over-period comparison)
5. Edge Case Testing (unit tests for data anomalies)

## Part 1: Data Generation
Generating 4 CSV files (orders, order_items, products, customers) with sample data and intentional inconsistencies (missing customer IDs, negative quantities, mixed date formats, unnormalized product names, invalid emails).

In [ ]:
import pandas as pd
from src.generator import generate_datasets

# Generate sample datasets
generate_datasets(output_dir="data/raw", seed=42)

# Preview generated raw files
print("Orders Sample:")
display(pd.read_csv("data/raw/orders.csv").head())

print("Products Sample:")
display(pd.read_csv("data/raw/products.csv").head())

## Part 2: Data Cleaning & Integrity Check
Cleaning raw data by fixing date formats, trimming product names, finding invalid emails, and checking for orphan order items.

In [ ]:
import json
from src.cleaner import run_data_cleaning_pipeline

# Run data cleaner
cleaning_report = run_data_cleaning_pipeline(raw_dir="data/raw", cleaned_dir="data/cleaned")

# Display issue report summary
print("Cleaning Report Summary:")
print(json.dumps(cleaning_report, indent=2))

## Part 3: SQL Analysis (16 Analytical Queries)
Loading cleaned data into the database and executing SQL queries for business insights.

In [ ]:
from src.database import DatabaseManager
from src.sql_analytics import SQLAnalyticsRunner
import matplotlib.pyplot as plt

# Connect to database & load data
db_mgr = DatabaseManager()
db_mgr.connect()
db_mgr.create_tables()
db_mgr.load_cleaned_data(cleaned_dir="data/cleaned")

# Run all 16 analytical queries
sql_runner = SQLAnalyticsRunner(db_manager=db_mgr)
query_results = sql_runner.run_all_queries(queries_dir="queries")

### Visualization of Key Metrics
Plotting category revenue and top customers.

In [ ]:
# Plot Revenue per Category
df_cat = query_results.get('01_total_revenue_per_category')
if df_cat is not None and not df_cat.empty:
    plt.figure(figsize=(7, 4))
    plt.bar(df_cat['category'], df_cat['total_revenue'], color='skyblue')
    plt.title('Total Revenue by Category')
    plt.xlabel('Category')
    plt.ylabel('Revenue ($)')
    plt.show()

# Plot Top 10 Customers
df_cust = query_results.get('02_top_10_customers')
if df_cust is not None and not df_cust.empty:
    plt.figure(figsize=(8, 4))
    plt.barh(df_cust['customer_name'], df_cust['total_order_value'], color='lightgreen')
    plt.gca().invert_yaxis()
    plt.title('Top 10 Customers by Order Value')
    plt.xlabel('Total Revenue ($)')
    plt.show()

## Part 4: CLI Reporting Tool
Generating dynamic summary reports for a specified date range and comparing performance to the previous period.

In [ ]:
from src.cli import generate_cli_report

# Generate report summary
generate_cli_report(db_manager=db_mgr, report_type="monthly", start_date="2024-01-01", end_date="2024-12-31")

## Part 5: Edge Case Handling & Unit Tests
Testing edge cases including orphan order items, discount > 100%, 0 quantity, and future dates.

In [ ]:
from tests.test_edge_cases import (
    test_orphan_order_items,
    test_excessive_discount_percent,
    test_zero_quantity_items,
    test_future_order_date
)

# Run tests
test_orphan_order_items()
test_excessive_discount_percent()
test_zero_quantity_items()
test_future_order_date()
print("All test cases passed!")